# 03 - Feature Engineering, Model Training & F_0.5 Calibration

Trains a supervised LightGBM pairwise classifier using GroupKFold and optimizes the macro F_0.5 decision threshold.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, "..")

from src.config import DATA_DIR_TRAIN
from src.load_data import load_dataset_split
from src.blocking import MultiIndexBlocker
from src.features import generate_features_dataset
from src.train import EntityClassifier
from src.threshold import optimize_f05_threshold

## 1. Generate Candidates & Features

In [ ]:
df_s1, df_s2, df_s3, gt = load_dataset_split(DATA_DIR_TRAIN, "train")
blocker = MultiIndexBlocker()
candidates = blocker.generate_candidates(df_s1, df_s2, df_s3)

df_target = pd.concat([df_s2, df_s3], ignore_index=True)
df_pairs = generate_features_dataset(candidates, df_s1, df_target)

# Create ground truth binary labels
labels = [1 if row["target_id"] in gt.get(row["s1_id"], set()) else 0 for _, row in df_pairs.iterrows()]
df_pairs["label"] = labels
print(f"Candidate Pairs: {len(df_pairs)} ({sum(labels)} true matches)")

## 2. Train LightGBM with GroupKFold

In [ ]:
clf = EntityClassifier(model_type="lightgbm")
oof_probs, y = clf.train_cv(df_pairs, n_splits=5)

## 3. Threshold Optimization for Macro F_0.5

In [ ]:
best_thresh, best_f05, metrics = optimize_f05_threshold(df_pairs, oof_probs, gt)
print(f"Best Threshold: {best_thresh:.3f}")
print(f"Validation Macro F_0.5: {best_f05:.4f}")
print(f"Singleton Accuracy: {metrics['singleton_accuracy']:.4f}")